In [41]:
import ipywidgets
from ipywidgets import Output
from IPython.display import display, clear_output
from ipycanvas import Canvas, MultiCanvas

from random import *
import time
import threading

multi_canvas = MultiCanvas(4, width=500, height=500)
background = multi_canvas[0]
tree = multi_canvas[1]
player = multi_canvas[2]
coin = multi_canvas[3]

key_press = Output()

player_x = 0
player_y = 240
player_speed = 10

coin_x = randint(10, 490)
coin_y = randint(10, 490)
coin_num = 0
coin_win = 20

trees = []
scroll_speed = 8
alive = True
win = False

def setting():
    """
    Functions generates a simple drawing of a grass background.
    """
    background.fill_style = "YellowGreen"
    background.fill_rect(0, 0, 500, 500)

def tree_draw(tree_x, tree_y):
    """
    Functions generates a simple drawing of a tree.
    
    Parameters
    ----------
    tree_x: Integer
        The x value of the tree's coordinates on the canvas
    tree_y: Integer
        The y value of the tree's coordinates on the canvas
    """
    tree.fill_style = "SaddleBrown"
    tree.fill_rect(tree_x, tree_y, 16, 40)
    tree.fill_style = "ForestGreen"
    tree.fill_circle(tree_x + 8, tree_y, 20)

def player_draw():
    """
    Function drwas the image of the player character as a simple
    box using the global player x and y variables.
    """
    player.fill_style = "White"
    player.fill_rect(player_x, player_y, 20, 20)

def draw_coin():
    """
    Function draws a simple coin using the global coin x and y 
    variables and displays a coin count in the top left corner.
    """
    coin.fill_style = "Yellow"
    coin.fill_circle(coin_x, coin_y, 10)
    coin.fill_style = "Black"
    coin.font = "16px serif"
    coin.fill_text(f"Coins: {coin_num}", 10, 10)

@key_press.capture()
def player_move(key, shift_key, ctrl_key, meta_key):
    global player_x, player_y
    player.clear()
    if key == "ArrowRight" and player_x < 480:
        player_x += player_speed
    if key == "ArrowLeft" and player_x > 0:
        player_x -= player_speed
    if key == "ArrowUp" and player_y > 0:
        player_y -= player_speed
    if key == "ArrowDown" and player_y < 480:
        player_y += player_speed
    player_draw()

def questions():
    """
    Function asks the user questions for them to inut an answer 
    which will then be used to determing the difficulty of the 
    game and amount of coins needed to win.
    """
    global player_speed, scroll_speed, coin_win
    difficulty = str(input("What difficulty would you like to play at, easy, medium, or hard? "))
    if difficulty == "easy" or difficulty == "Easy":
        scroll_speed = 8
    elif difficulty == "medium" or difficulty == "Medium":
        scroll_speed = 13
    elif difficulty == "hard" or difficulty == "Hard":
        scroll_speed = 20
        user_speed_input = str(input("Would you like to increase player speed as well? "))
        if user_speed_input == "yes" or user_speed_input == "Yes":
            player_speed += 5
        elif user_speed_input == "no" or user_speed_input == "No":
            player_speed = player_speed
        else:
            print("Refresh and plaese answer yes or no.")
    else:
        print("Please refresh and input a valid opption.")
    user_coin_input = int(input("How many coins would you like to collect? Please make it 50 or under. "))
    if user_coin_input > 50:
        print("That's too many for one trip! Refresh and input a whole number less than 50.")
    elif user_coin_input <= 0:
        print("Stop kidding. Refresh and input a whole number bigger than 0.")
    else:
        coin_win = user_coin_input

def coin_count():
    """
    Function checks if the player has collided with the coin and if so,
    adds increases the counter for the number of coins collected, as well
    as resets the coin. It also provides the conditional to reset the coin
    if it goes off-screen.
    """
    global coin_x, coin_y, coin_num
    # Collision check
    if coin_x in range(player_x, (player_x + 20)) and coin_y in range(player_y, (player_y + 20)):
        coin_num += 1
        coin.clear()
        coin_x = 490
        coin_y = randint(10, 490)
        draw_coin()
    # Reset if offscreen
    if coin_x <= -10:
        coin.clear()
        coin_x = 490
        coin_y = randint(10, 490)
        draw_coin()

def win_condition():
    """
    Function creates the condition for the player to win the game and 
    generates the screen that will be shown when they do.
    """
    global alive, win, coin_num, coin_win
    # If all coins collected, show win screen
    if coin_num >= coin_win:
        win = True
        alive = False
        # Clear all layers
        background.clear()
        tree.clear()
        player.clear()
        coin.clear()
        # Win screen
        background.fill_style = "Black"
        background.fill_rect(0, 0, 500, 500)
        background.fill_style = "White"
        background.font = "48px serif"
        background.fill_text("🎉 YOU WIN! 🎉", 65, 200)
        background.fill_text(f"Score: {coin_num} coins", 110, 260)
        background.font = "20px serif"
        background.fill_text("Refresh to play again", 160, 320)
        
def loss_condition():
    """
    Function checks for collision between the player and all trees.
    If hit, it triggers a game-over screen.
    """
    global alive
    for (tree_x, tree_y) in trees:
        # Collision check
        player_area = (player_x, player_y, 20, 20)
        tree_area = (tree_x, tree_y, 16, 40)
        tree_top_area = (tree_x - 12, tree_y - 20, 40, 40)  # Canopy area
        if collision_area(player_area, tree_area) or collision_area(player_area, tree_top_area):
            alive = False
    # If hit tree, show loss screen
    if alive == False and win == False:
        background.clear()
        tree.clear()
        player.clear()
        coin.clear()
        background.fill_style = "Black"
        background.fill_rect(0, 0, 500, 500)
        background.fill_style = "Red"
        background.font = "48px serif"
        background.fill_text("You Lose!", 145, 200)
        background.fill_text(f"Score: {coin_num} coins", 110, 260)
        background.font = "20px serif"
        background.fill_text("Refresh to play again", 160, 320)

def collision_area(area1, area2):
    """
    Function determines the player and tree areas that are
    used to determine collision.
    
    Returns
    -------
    Function returns the area in which the player will be
    considered to have collided with a tree and lost.
    """
    x1, y1, width1, height1 = area1
    x2, y2, width2, height2 = area2
    return (
        x1 < x2 + width2 and
        x1 + width1 > x2 and
        y1 < y2 + height2 and
        y1 + height1 > y2
    )
        
def game_loop():
    """
    Function creates the logic that allows for the trees and coin 
    generated in the tree_draw and coin_draw functions to scroll 
    across the screen and reset when they've gone off-screen. It
    also includes the win and loss condition for the game.
    """
    while alive:
        global trees
        tree.clear()
        new_list = []
        for (tree_x, tree_y) in trees:
            tree_x -= scroll_speed
            # Reset if off screen
            if tree_x < -30:
                tree_x = 500
                tree_y = randint(20, 460)
            new_list.append((tree_x, tree_y))
            tree_draw(tree_x, tree_y)
        trees = new_list
        global coin_x
        coin.clear()
        coin_x -= scroll_speed
        draw_coin()
        coin_count()
        win_condition()
        loss_condition()
        time.sleep(1)

def main():
    """
    Function contains the logic of the entire program, allowing for the
    game to run and be played.
    """
    global trees
    questions()
    setting()
    # Create 4 trees
    trees = [
        (randint(50, 120), randint(170, 225)),
        (randint(170, 225), randint(100, 295)),
        (randint(295, 370), randint(255, 420)),
        (randint(420, 480), randint(100, 295)),
    ]
    for (tree_x, tree_y) in trees:
        tree_draw(tree_x, tree_y)
    draw_coin()
    player_draw()
    # Start tree and coin scroll thread
    threading.Thread(target = game_loop, daemon = True).start()

main()

multi_canvas.on_key_down(player_move)
display(multi_canvas, key_press)



MultiCanvas(width=500)

Output()